In [1]:
# IMPORT PACKAGES

from pathlib import Path

import pandas as pd
import rasterio
from tqdm import tqdm

In [2]:

# ROOT DIRECTORY

ROOT_DIR = Path("..")

# INPUT DATASET
DATASET_DIR = ROOT_DIR / "dataset_clipped"

# OUTPUTS
OUTPUTS_DIR = ROOT_DIR / "outputs"

# Create outputs folder
OUTPUTS_DIR.mkdir(exist_ok=True)

print("Paths ready.")


Paths ready.


In [3]:
# FIND ALL TIFF FILES

tif_files = list(DATASET_DIR.rglob("*.tif"))

print(f"Found {len(tif_files)} tif files.")

Found 120 tif files.


In [4]:
def validate_rasters():

    validation_results = []

    for tif_path in tqdm(tif_files):

        try:

            with rasterio.open(tif_path) as src:

                result = {

                    "file_name": tif_path.name,
                    "year": tif_path.parts[-3],
                    "flight": tif_path.parts[-2],

                    "crs": str(src.crs),

                    "width": src.width,
                    "height": src.height,

                    "bands": src.count,

                    "pixel_size_x": src.res[0],
                    "pixel_size_y": src.res[1],

                    "dtype": src.dtypes[0],

                    "nodata": src.nodata,

                    "bounds": str(src.bounds),

                    "status": "OK"

                }

        except Exception as e:

            result = {

                "file_name": tif_path.name,
                "year": "UNKNOWN",
                "flight": "UNKNOWN",

                "crs": None,

                "width": None,
                "height": None,

                "bands": None,

                "pixel_size_x": None,
                "pixel_size_y": None,

                "dtype": None,

                "nodata": None,

                "bounds": None,

                "status": f"ERROR: {e}"

            }

        validation_results.append(result)

    return pd.DataFrame(validation_results)

In [5]:
validation_df = validate_rasters()

validation_df.head()

100%|████████████████████████████████████████████████████████████████████████████████| 120/120 [00:02<00:00, 59.95it/s]


,file_name,year,flight,crs,width,height,bands,pixel_size_x,pixel_size_y,dtype,nodata,bounds,status
0,2025_04_09_thermal_transparent_reflectance_gra...,2025,flight_2025_04_09,EPSG:32632,1064,1417,1,0.08270,0.08270,float32,-10000.0,"BoundingBox(left=354191.29104000004, bottom=57...",OK
1,multispectral_2025_04_09_transparent_reflectan...,2025,flight_2025_04_09,EPSG:32632,3156,4204,1,0.02786,0.02786,float32,-10000.0,"BoundingBox(left=354191.30733000004, bottom=57...",OK
2,multispectral_2025_04_09_transparent_reflectan...,2025,flight_2025_04_09,EPSG:32632,3156,4204,3,0.02786,0.02786,float32,-10000.0,"BoundingBox(left=354191.30733000004, bottom=57...",OK
3,multispectral_2025_04_09_transparent_reflectan...,2025,flight_2025_04_09,EPSG:32632,3156,4204,1,0.02786,0.02786,float32,-10000.0,"BoundingBox(left=354191.30733000004, bottom=57...",OK
4,multispectral_2025_04_09_transparent_reflectan...,2025,flight_2025_04_09,EPSG:32632,3156,4204,1,0.02786,0.02786,float32,-10000.0,"BoundingBox(left=354191.30733000004, bottom=57...",OK


In [6]:
# SAVE CSV

output_csv = OUTPUTS_DIR / "raster_validation.csv"

validation_df.to_csv(output_csv, index=False)

print(f"Validation CSV saved:\n{output_csv}")

Validation CSV saved:
outputs\raster_validation.csv
